In [ ]:
import re
import glob
import argparse
import matplotlib.pyplot as plt
from collections import defaultdict

In [ ]:
def parse_log_line(line):
    """解析单行日志，提取 step 和所有指标"""
    # 移除 ANSI 颜色代码
    ansi_escape = re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])|\^\[\[\d+m')
    line = ansi_escape.sub('', line)
    
    # 匹配 step:数字
    step_match = re.search(r'step:(\d+)', line)
    if not step_match:
        return None, {}
    
    step = int(step_match.group(1))
    
    # 提取所有 key:value 对
    metrics = {}
    # 匹配形如 "metric/name:value" 或 "val-aux/name@1:value" 的模式
    # 支持 - @ 等特殊字符
    pattern = r'([a-zA-Z_][a-zA-Z0-9_/\-@]*):(-?[\d.]+(?:e[+-]?\d+)?)'
    matches = re.findall(pattern, line)
    
    for key, value in matches:
        if key != 'step':  # 排除 step 本身
            try:
                metrics[key] = float(value)
            except ValueError:
                pass
    
    return step, metrics

In [ ]:
def parse_log_file(filepath):
    """解析整个日志文件"""
    data = defaultdict(list)
    steps = []
    
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            step, metrics = parse_log_line(line)
            if step is not None and metrics:
                steps.append(step)
                for key, value in metrics.items():
                    data[key].append((step, value))
    
    return data

In [ ]:
def plot_metrics(data, metrics_to_plot, output_file=None, title=None):
    """绘制指定指标的曲线"""
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for metric in metrics_to_plot:
        if metric in data:
            points = sorted(data[metric], key=lambda x: x[0])
            steps = [p[0] for p in points]
            values = [p[1] for p in points]
            ax.plot(steps, values, marker='o', markersize=3, label=metric, linewidth=1.5)
        else:
            print(f"警告: 指标 '{metric}' 未在日志中找到")
            print(f"可用指标: {list(data.keys())[:20]}...")
    
    ax.set_xlabel('Step', fontsize=12)
    ax.set_ylabel('Value', fontsize=12)
    ax.set_title(title or 'Training Curves', fontsize=14)
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if output_file:
        plt.savefig(output_file, dpi=150, bbox_inches='tight')
        print(f"图表已保存到: {output_file}")
    
    plt.show()

In [ ]:
log_files = ["multiturn_grpo_v4_2_*"]

# 展开通配符
all_files = []
for pattern in log_files:
    matched = glob.glob(pattern)
    if matched:
        all_files.extend(matched)
    else:
        all_files.append(pattern)  # 保留原始路径以便报错

# 合并所有日志文件的数据
combined_data = defaultdict(list)

for filepath in all_files:
    try:
        print(f"正在解析: {filepath}")
        data = parse_log_file(filepath)
        for key, values in data.items():
            combined_data[key].extend(values)
    except FileNotFoundError:
        print(f"错误: 文件不存在 - {filepath}")
    except Exception as e:
        print(f"错误: 解析文件 {filepath} 时出错 - {e}")

if not combined_data:
    print("未能从日志文件中解析出任何数据")

# 列出所有可用指标
if False:
    print("\n可用指标:")
    for i, metric in enumerate(sorted(combined_data.keys()), 1):
        print(f"  {i:3d}. {metric}")

metrics = ['critic/rewards/mean']
title = "train_reward_step"
output_dir = "figs/" + title


# 绘制曲线
plot_metrics(combined_data, metrics, output_dir, title)

metrics = ["val-aux/multiturnnd/score/mean@1"]
title = "valid_scores_mean@1"
output_dir = "figs/" + title

plot_metrics(combined_data, metrics, output_dir, title)


In [ ]:
metrics = ["val-aux/multiturnnd/turn_reward/mean@1"]
title = "valid_turn_reward_mean@1"
output_dir = "figs/" + title

plot_metrics(combined_data, metrics, output_dir, title)

metrics = ["val-aux/multiturnnd/final_reward/mean@1"]
title = "valid_final_reward_mean@1"
output_dir = "figs/" + title

plot_metrics(combined_data, metrics, output_dir, title)